# Day 5: Data Manipulation and Analysis
## Notebook 2: Data Cleaning and Transformation

**Introduction to Scientific Programming**  
CNC-UC

This notebook covers:
- Handling missing data
- Detecting and handling outliers
- Data type conversions
- String and categorical operations
- Creating new columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

%matplotlib inline

print("Setup complete!")

## 1. Handling Missing Data

### Creating dataset with missing values

In [ ]:
# Create example dataset with missing values
np.random.seed(42)
n = 50

data_with_na = pd.DataFrame({
    'subject_id': [f'S{i:02d}' for i in range(1, n+1)],
    'age': np.random.randint(20, 40, n),
    'reaction_time': np.random.gamma(2, 200, n),
    'accuracy': np.random.random(n),
    'condition': np.random.choice(['control', 'drug_A', 'drug_B'], n)
})

# Introduce missing values
data_with_na.loc[np.random.choice(data_with_na.index, 5), 'reaction_time'] = np.nan
data_with_na.loc[np.random.choice(data_with_na.index, 3), 'age'] = np.nan
data_with_na.loc[np.random.choice(data_with_na.index, 7), 'accuracy'] = np.nan

print("Dataset with missing values:")
data_with_na.head(10)

### Detecting missing values

In [ ]:
# Count missing values per column
print("Missing values per column:")
print(data_with_na.isnull().sum())
print()

# Percentage missing
print("Percentage missing per column:")
print((data_with_na.isnull().sum() / len(data_with_na) * 100).round(2))
print()

# Rows with any missing values
rows_with_na = data_with_na[data_with_na.isnull().any(axis=1)]
print(f"Rows with any missing values: {len(rows_with_na)}")
print(rows_with_na.head())

### Deletion strategies

In [ ]:
# Drop rows with ANY missing value
df_dropna_any = data_with_na.dropna()
print(f"Original rows: {len(data_with_na)}")
print(f"After dropna(): {len(df_dropna_any)}")
print(f"Rows lost: {len(data_with_na) - len(df_dropna_any)}")
print()

# Drop rows with ALL missing values
df_dropna_all = data_with_na.dropna(how='all')
print(f"After dropna(how='all'): {len(df_dropna_all)}")
print()

# Drop rows with missing in specific columns
df_dropna_subset = data_with_na.dropna(subset=['reaction_time', 'accuracy'])
print(f"After dropna(subset=['reaction_time', 'accuracy']): {len(df_dropna_subset)}")
print()

# Threshold: require minimum non-NA values
df_dropna_thresh = data_with_na.dropna(thresh=4)  # At least 4 non-NA
print(f"After dropna(thresh=4): {len(df_dropna_thresh)}")

### Imputation strategies

In [ ]:
# Create a clean copy for imputation examples
df_imputed = data_with_na.copy()

# Fill with specific value
df_imputed['accuracy_filled_zero'] = df_imputed['accuracy'].fillna(0)

# Fill with mean
df_imputed['rt_filled_mean'] = df_imputed['reaction_time'].fillna(
    df_imputed['reaction_time'].mean()
)

# Fill with median
df_imputed['rt_filled_median'] = df_imputed['reaction_time'].fillna(
    df_imputed['reaction_time'].median()
)

# Forward fill (propagate previous value)
df_imputed['rt_ffill'] = df_imputed['reaction_time'].ffill()

# Backward fill
df_imputed['rt_bfill'] = df_imputed['reaction_time'].bfill()

print("Imputation comparison:")
df_imputed[['reaction_time', 'rt_filled_mean', 'rt_filled_median', 
                  'rt_ffill', 'rt_bfill']].head(15)

### Interpolation for timeseries

In [ ]:
# Create timeseries with gaps
time_index = pd.date_range('2024-01-01', periods=20, freq='1h')
signal = np.sin(np.linspace(0, 4*np.pi, 20)) + np.random.normal(0, 0.1, 20)

# Introduce gaps
signal[[3, 7, 12, 15]] = np.nan

ts_data = pd.DataFrame({
    'signal': signal
}, index=time_index)

# Different interpolation methods
ts_data['linear'] = ts_data['signal'].interpolate(method='linear')
ts_data['polynomial'] = ts_data['signal'].interpolate(method='polynomial', order=2)

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ts_data['signal'].plot(ax=axes[0], marker='o', title='Original with NaN')
ts_data['linear'].plot(ax=axes[1], marker='o', title='Linear Interpolation')
ts_data['polynomial'].plot(ax=axes[2], marker='o', title='Polynomial Interpolation')

plt.tight_layout()
plt.show()

## 2. Detecting and Handling Outliers

In [ ]:
# Create dataset with outliers
np.random.seed(42)
normal_data = np.random.normal(500, 50, 95)
outliers = np.array([150, 200, 900, 950, 1000])  # Clear outliers
data_with_outliers = np.concatenate([normal_data, outliers])

df_outliers = pd.DataFrame({
    'reaction_time': data_with_outliers,
    'subject': [f'S{i:02d}' for i in range(1, 101)]
})

print("Data with outliers:")
df_outliers.describe()

### Z-score method

In [ ]:
# Calculate z-scores
df_outliers['z_score'] = np.abs(stats.zscore(df_outliers['reaction_time']))

# Flag outliers (|z| > 3)
df_outliers['is_outlier_z'] = df_outliers['z_score'] > 3

print("Outliers detected with z-score:")
print(df_outliers[df_outliers['is_outlier_z']])
print(f"\nNumber of outliers: {df_outliers['is_outlier_z'].sum()}")

### IQR method

In [ ]:
# Calculate IQR
Q1 = df_outliers['reaction_time'].quantile(0.25)
Q3 = df_outliers['reaction_time'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Flag outliers
df_outliers['is_outlier_iqr'] = (
    (df_outliers['reaction_time'] < lower_bound) | 
    (df_outliers['reaction_time'] > upper_bound)
)

print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")
print(f"\nNumber of outliers (IQR): {df_outliers['is_outlier_iqr'].sum()}")

### Visual detection

In [ ]:
# Boxplot shows outliers automatically
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df_outliers.boxplot(column='reaction_time', ax=axes[0])
axes[0].set_title('Boxplot (outliers marked automatically)')
axes[0].set_ylabel('Reaction Time (ms)')

df_outliers['reaction_time'].hist(bins=30, ax=axes[1])
axes[1].axvline(lower_bound, color='r', linestyle='--', label='IQR bounds')
axes[1].axvline(upper_bound, color='r', linestyle='--')
axes[1].set_title('Histogram with IQR bounds')
axes[1].set_xlabel('Reaction Time (ms)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Data Type Conversions

In [ ]:
# Create example data
df_types = pd.DataFrame({
    'subject_id': ['1', '2', '3', '4', '5'],
    'trial': ['10', '20', '30', '40', '50'],
    'correct': ['1', '0', '1', '1', '0'],
    'dose': ['low', 'medium', 'high', 'low', 'medium']
})

print("Original dtypes:")
df_types.dtypes

### Basic conversions

In [ ]:
# Convert to numeric
df_types['subject_id'] = df_types['subject_id'].astype(int)
df_types['trial'] = df_types['trial'].astype(int)
df_types['correct'] = df_types['correct'].astype(bool)

print("After conversion:")
print(df_types.dtypes)
df_types

### Safe conversions

In [ ]:
# Data with some non-numeric values
messy_data = pd.Series(['100', '200', 'NA', '300', 'error', '400'])

# This would fail: messy_data.astype(int)

# Safe conversion with errors='coerce'
clean_data = pd.to_numeric(messy_data, errors='coerce')
print("Safe conversion (invalid → NaN):")
print(clean_data)
print()

# Find the offending values instead of hiding them
invalid = messy_data[clean_data.isna()]
print("Values that could not be converted:")
print(invalid)

### Categorical data for memory efficiency

In [ ]:
# Create large dataset with repeated values
large_df = pd.DataFrame({
    'condition': np.random.choice(['control', 'drug_A', 'drug_B'], 10000)
})

# Memory usage before
memory_before = large_df['condition'].memory_usage(deep=True)
print(f"Memory usage (object): {memory_before / 1024:.2f} KB")

# Convert to category
large_df['condition'] = large_df['condition'].astype('category')
memory_after = large_df['condition'].memory_usage(deep=True)
print(f"Memory usage (category): {memory_after / 1024:.2f} KB")
print(f"Reduction: {(1 - memory_after/memory_before)*100:.1f}%")

## 4. Working with Categorical and String Data

### String operations (.str accessor)

In [ ]:
# Create example string data
df_str = pd.DataFrame({
    'gene_id': ['  BDNF_human  ', 'TrkB_MOUSE', 'creb_rat', 'ARC_Human'],
    'description': ['brain-derived neurotrophic factor', 
                    'tropomyosin receptor kinase B',
                    'cAMP response element-binding protein',
                    'activity regulated cytoskeleton']
})

print("Original data:")
df_str

In [ ]:
# String manipulation
df_str['gene_clean'] = df_str['gene_id'].str.strip()  # Remove whitespace
df_str['gene_upper'] = df_str['gene_id'].str.upper()  # Uppercase
df_str['gene_lower'] = df_str['gene_id'].str.lower()  # Lowercase
df_str['gene_replaced'] = df_str['gene_id'].str.replace('_', '-')  # Replace

print("After string operations:")
df_str[['gene_id', 'gene_clean', 'gene_upper', 'gene_lower']]

In [ ]:
# Extraction and splitting
df_str['gene_name'] = df_str['gene_clean'].str.split('_').str[0]
df_str['species'] = df_str['gene_clean'].str.split('_').str[1]
df_str['desc_length'] = df_str['description'].str.len()
df_str['contains_kinase'] = df_str['description'].str.contains('kinase')

print("\nExtraction results:")
df_str[['gene_name', 'species', 'desc_length', 'contains_kinase']]

### Categorical operations

In [ ]:
# Create ordered categorical
df_cat = pd.DataFrame({
    'dose': ['low', 'high', 'medium', 'low', 'high', 'medium']
})

# Convert to ordered category
df_cat['dose'] = pd.Categorical(
    df_cat['dose'],
    categories=['low', 'medium', 'high'],
    ordered=True
)

print("Ordered categorical:")
print(df_cat['dose'])
print()

# Now we can sort properly
print("Sorted by dose:")
print(df_cat.sort_values('dose'))

## 5. Creating New Columns - Transformations

In [ ]:
# Create example dataset
np.random.seed(42)
df_transform = pd.DataFrame({
    'expression': np.random.exponential(100, 20),
    'control': np.random.exponential(100, 20),
    'treated': np.random.exponential(120, 20),
    'threshold': np.random.uniform(50, 150, 20)
})

print("Original data:")
df_transform.head()

### Simple arithmetic operations

In [ ]:
# Log transformation
df_transform['log_expression'] = np.log2(df_transform['expression'] + 1)

# Combining columns
df_transform['fold_change'] = df_transform['treated'] / df_transform['control']

# Normalization
mean_expr = df_transform['expression'].mean()
std_expr = df_transform['expression'].std()
df_transform['z_score'] = (df_transform['expression'] - mean_expr) / std_expr

print("After transformations:")
df_transform[['expression', 'log_expression', 'fold_change', 'z_score']].head()

### Conditional values

In [ ]:
# Simple condition with np.where
df_transform['category'] = np.where(
    df_transform['expression'] > 100, 
    'high', 
    'low'
)

# Multiple conditions with np.select
conditions = [
    df_transform['expression'] < 50,
    (df_transform['expression'] >= 50) & (df_transform['expression'] < 100),
    df_transform['expression'] >= 100
]
values = ['low', 'medium', 'high']
df_transform['category_3level'] = np.select(conditions, values, default='unknown')

print("\nConditional categorization:")
df_transform[['expression', 'category', 'category_3level']].head(10)

### Custom functions with .apply()

In [ ]:
# Custom function
def classify_expression(value):
    if value < 50:
        return 'very_low'
    elif value < 100:
        return 'low'
    elif value < 150:
        return 'medium'
    else:
        return 'high'

df_transform['custom_category'] = df_transform['expression'].apply(classify_expression)

# Lambda function
df_transform['doubled'] = df_transform['expression'].apply(lambda x: x * 2)

# Mapping dictionary
category_map = {'low': 1, 'medium': 2, 'high': 3}
df_transform['category_numeric'] = df_transform['category_3level'].map(category_map)

print("\nCustom functions and mapping:")
df_transform[['expression', 'custom_category', 'doubled', 'category_numeric']].head()

## Summary

In this notebook we covered:
- Detecting and handling missing data (deletion, imputation, interpolation)
- Outlier detection (z-score, IQR methods)
- Data type conversions (safe conversions, categorical types)
- String operations and categorical data
- Creating new columns (arithmetic, conditional, apply)

**Next notebook:** Grouping, Aggregation, and Reshaping

In [ ]:
print("Notebook 2 complete!")